# Workshop 2 — nn.Module, Activations, and Optimisers

Last session, we built and trained a neural network using only PyTorch tensors and autograd. This involved hand-writing the parameters, the forward pass, the loss, and the update loop.

Today we replace each of those hand-written pieces with PyTorch's built-in abstractions.

## Part 0 — Setup

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

torch.manual_seed(0)
print("torch", torch.__version__)

## Part 1 — Recap: the from-scratch network

Here is exactly what we built last session, condensed. A 2-layer network fitting a noisy sine curve, with every piece written out by hand. Run it and keep the final loss in mind — we'll reproduce it with abstractions.

In [ ]:
# Generate synthetic data
N = 200
X = torch.linspace(-3, 3, N).unsqueeze(1)
y = torch.sin(X) + 0.1 * torch.randn(N, 1)

# Define parameters by hand using Kaiming initialisation
torch.manual_seed(1)
hidden = 64

W1 = (torch.randn(1, hidden) * math.sqrt(2/1)).requires_grad_(True)
b1 = torch.zeros(hidden, requires_grad=True)
W2 = (torch.randn(hidden, 1) * math.sqrt(2/hidden)).requires_grad_(True)
b2 = torch.zeros(1, requires_grad=True)
params = [W1, b1, W2, b2]

# Training loop
lr = 0.01
scratch_losses = []
for epoch in range(2000):
    ## Forward pass
    pre_activation = X @ W1 + b1
    post_activation = torch.relu(pre_activation)
    y_hat = post_activation @ W2 + b2

    ## Loss
    loss = ((y_hat - y) ** 2).mean()

    ## Backward pass
    loss.backward()

    ## Update
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad

    ## Zero gradients
    for p in params:
        p.grad.zero_()

    ## Store loss
    scratch_losses.append(loss.item())

print(f"from-scratch final loss: {scratch_losses[-1]:.4f}")

## Part 2 — `nn.Module`: the core abstraction

 A module:
- holds parameters (it tracks them for you, needing no manual lists),
- defines a `forward` method,
- is callable. `model(x)` runs `forward(x)`.

Let's rebuild the same network as a custom `nn.Module`. Notice `nn.Linear(in, out)` is the `xW + b` operation. It creates and tracks `W` and `b` for you.

In [ ]:
class TwoLayerNet(nn.Module):
    def __init__(self, in_dim=1, hidden=64, out_dim=1):
        super().__init__()                       
        self.fc1 = nn.Linear(in_dim, hidden)     # creates W1, b1
        self.fc2 = nn.Linear(hidden, out_dim)    # creates W2, b2
        self.act = nn.ReLU()

    def forward(self, x):
        h = self.act(self.fc1(x))
        return self.fc2(h)

# Create a model by instantiating the TwoLayerNet
model = TwoLayerNet()

# We can inspect the model through print
print(model)

# We can extract the parameter tensors
print("\nParameters PyTorch is tracking for us:")
for name, p in model.named_parameters():
    print(f"  {name:12s} shape {tuple(p.shape)}")

### `nn.Sequential`: an even shorter way

When a network is just a straight collections of layers, `nn.Sequential` saves you writing a class. The following is equivalent to the `TwoLayerNet` above.

In [ ]:
model_seq = nn.Sequential(nn.Linear(1, 64),
                          nn.ReLU(),
                          nn.Linear(64, 1))
print(model_seq)

My advice on when to use which:

- `nn.Sequential` when you just want to build a neural network with a standard linear stacks of layers.
- Custom `nn.Module` when you need branching, skip connections, multiple inputs/outputs, or any logic custom in `forward`.

## Part 3 — `nn.MSELoss` and `optim`

Two more hand-written pieces to abstract:
- **Loss:** `nn.MSELoss()` is exactly `((y_hat - y)**2).mean()`.
- **Optimiser:** `optim.SGD(model.parameters(), lr=...)` knows about all the model's parameters, so `optimizer.step()` does the update and `optimizer.zero_grad()` clears the gradients.

Here is the entire training loop with abstractions.

In [ ]:
def train(model, X, y, n_epochs, optimiser, loss_fn):
    losses = []

    for epoch in range(n_epochs):
        y_hat = model(X)               # 1. forward
        loss = loss_fn(y_hat, y)       # 2. loss
        optimiser.zero_grad()          # 3. zero
        loss.backward()                # 4. backward
        optimiser.step()               # 5. step
        losses.append(loss.item())

    return losses

In [ ]:
torch.manual_seed(1)

model = TwoLayerNet()
opt_sgd = torch.optim.SGD(model.parameters(), lr=0.01)
mse_loss = nn.MSELoss()

train_losses = train(model, X, y, n_epochs = 2000, optimiser = opt_sgd, loss_fn = mse_loss)

print(f"nn.Module final loss: {train_losses[-1]:.4f}")
print(f"from-scratch  (Part 1): {scratch_losses[-1]:.4f}")

Both fit the same curve with the same algorithm. The exact numbers differ slightly because nn.Linear uses its own default initialisation, but the loop is now 5 lines instead of 15.

## Part 4 — Activation functions and their effects

PyTorch gives us many loss functions that we can try out. Below are some of the more commonly used ones for training modern neural networks.

In [ ]:
# Define input values for plotting activation functions
z = torch.linspace(-4, 4, 200)

activations = {
    "ReLU":      nn.ReLU()(z),
    "LeakyReLU": nn.LeakyReLU(0.1)(z),
    "GELU":      nn.GELU()(z),
    "Tanh":      nn.Tanh()(z),
    "Sigmoid":   nn.Sigmoid()(z),
}

fig, ax = plt.subplots(figsize=(7, 4.5))

for name, values in activations.items():
    ax.plot(z, values, linewidth=2.2, label=name)

ax.axhline(0, color="grey", linewidth=0.6)
ax.axvline(0, color="grey", linewidth=0.6)

ax.set_xlabel("z")
ax.set_ylabel("activation(z)")
ax.set_title("Common activation functions")

ax.legend()

plt.tight_layout()
plt.show()

- **ReLU** — flat then linear. It's cheap to compute, but has a hard kink at z=0, and zero gradient for `z < 0`. This can cause "dead units".
- **LeakyReLU** — like ReLU but has a small slope for `z < 0`, so units never fully die.
- **GELU** — a smooth version of ReLU, which has become the default in modern transformers.
- **Tanh** — smooth function which outputs in (−1, 1). However, it saturates at the extreme values of `z`.
- **Sigmoid** — smooth function which outputs in (0, 1). It also saturates at the extreme values of `z`. It's mostly used on the output layer for probabilities, not between hidden layers.

### 4.1 Effect on the fit and on training

We train the same network with each activation and compare.

In [ ]:
act_layers = {
    "ReLU":      nn.ReLU(),
    "LeakyReLU": nn.LeakyReLU(0.1),
    "GELU":      nn.GELU(),
    "Tanh":      nn.Tanh()
}

Xs = torch.linspace(-3, 3, 400).unsqueeze(1)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for idx, (name, layer) in enumerate(act_layers.items()):
    torch.manual_seed(1)
    # Define model, optimiser, loss function and train
    model    = nn.Sequential(nn.Linear(1, 64),
                             layer,
                             nn.Linear(64, 1))

    opt_adam = torch.optim.Adam(model.parameters(), lr=0.01)
    mse_loss = nn.MSELoss()
    losses   = train(model, X, y, n_epochs = 2000, optimiser=opt_adam, loss_fn=mse_loss)

    ax_fit = axes[0, idx]
    ax_fit.scatter(X, y, s=6, alpha=0.4)
    with torch.no_grad():
        ax_fit.plot(Xs, model(Xs), color="red", lw=2.2)
    ax_fit.set_title(f"{name}  (hidden=8)\nfinal loss: {losses[-1]:.4f}")
    ax_fit.set_xticks([])
    ax_fit.set_yticks([])

    ax_loss = axes[1, idx]
    ax_loss.plot(losses, color="blue", lw=1.5)
    ax_loss.set_yscale("log")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_title(f"{name} loss")
    if idx == 0:
        ax_loss.set_ylabel("Loss (log)")

plt.tight_layout()
plt.show()

- With only 8 units the contrast is clear, both **ReLU and LeakyReLU** produce visibly **kinked, piecewise-linear** fits, while **GELU and Tanh** trace **smooth** curves.
- **ReLU/GELU** dominate deep networks because they avoid the vanishing gradient problem. **Tanh** and **Sigmoid** have near-zero gradient when saturated, and stacking many such layers multiplies those small gradients toward zero. ReLU's gradient is 1 for positive inputs, so signal flows.

## Part 5 — Optimisers and their effects

In the previous lecture, we looked at full-batch gradient desecent. Today, we saw how in practice, stochastic/mini-batch gradient descen (SGD) is prefered. Two further improvements exist ontop of SGD:

- **SGD + momentum** — accumulates a running average of past gradients. This has the effect of accelerating parameter trajectories in the directions of consistent gradient directions, smoothing out perturbations.
- **Adam** — adapts the step size per parameter using running averages of the gradient and its square. Usually trains fastest with the least tuning.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

optimisers = [
    ("SGD",      lambda params: torch.optim.SGD(params, lr=0.01)),
    ("Momentum", lambda params: torch.optim.SGD(params, lr=0.01, momentum=0.9)),
    ("Adam",     lambda params: torch.optim.Adam(params, lr=0.01)),
]

for name, optimiser in optimisers:
    torch.manual_seed(1)
    model     = nn.Sequential(nn.Linear(1, 64),
                              nn.ReLU(), 
                              nn.Linear(64, 1))

    optimiser = optimiser(model.parameters())
    mse_loss  = nn.MSELoss()
    losses    = train(model, X, y, n_epochs=2000, optimiser=optimiser, loss_fn=mse_loss)
    
    ax.plot(losses, label=f"{name}  (final {losses[-1]:.4f})", lw=2)

ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (log)")
ax.set_title("Optimiser comparison")
ax.legend()
plt.tight_layout()
plt.show()

- Plain **SGD** descends slowest.
- **SGD+Momentum** is faster and smoother.
- **Adam** reaches convergence the quickest and smoothest.

### 5.1 The learning rate matters as much as the optimiser

Let's now see the effect of the learning rate on SGD performance when keeping everything else fixed. To make the curves clearer to see, we'll use a wider network of two hidden layers, each with ReLU activation functions.

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(10, 6))

learning_rates = [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5]
colors = plt.cm.viridis_r(np.linspace(0, 1, len(learning_rates)))

def has_diverged(losses, threshold=1e3):
    arr = np.array(losses)
    return np.any(np.isnan(arr)) or np.any(np.isinf(arr)) or arr[-1] > threshold

for idx, (lr, color) in enumerate(zip(learning_rates, colors)):
    torch.manual_seed(1)
    model = nn.Sequential(nn.Linear(1, 128),
                          nn.ReLU(),
                          nn.Linear(128, 64),
                          nn.ReLU(),
                          nn.Linear(64, 1))

    optimiser = torch.optim.SGD(model.parameters(), lr=lr)
    mse_loss  = nn.MSELoss()
    losses    = train(model, X, y, n_epochs=3000, optimiser=optimiser, loss_fn=mse_loss)

    if has_diverged(losses):
        ax.plot([np.nan]*3000, label=f"lr={lr}  (diverged)", lw=2, ls='--', color=color, alpha=0.7)
    else:
        final = losses[-1]
        label = f"lr={lr}  (final {final:.4f})"
        ax.plot(losses, label=label, lw=2, color=color)

ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (log)")
ax.set_title("Learning-rate sweep (SGD) — Loss Curves")
ax.legend()
plt.tight_layout()
plt.show()

We see that for very low learning rates, training is incredible slow. Increasing the learning rate does improve training performance up till a point. Going from LR=0.05 to LR=0.1 seemed to in fact worsen training performance. Increasing it even more to LR=0.5 caused training to diverge.

## Recap

We replaced every hand-written piece of the from-scratch network with a PyTorch abstraction:

1. **`nn.Module`** holds parameters and defines `forward`. `nn.Sequential` is the shortcut for plain neural networks. Custom modules for anything with structure.
2. **`nn.MSELoss`** and **`torch.optim`** replace the manual loss and update loop.
3. There exist a large number activation functions: ReLU/GELU for hidden layers with favourable gradient flows, Tanh/GELUsmooth for smooth targets, sigmoid/softmax at the output.
4. Adam, which builds on from SGD + momentum, has become the state of the art optimiser to use in deep learning training.

### Next: Practical Workshop 2 — Dataset & DataLoader
We'll cover `Dataset`/`DataLoader` pattern that every PyTorch project uses for handling data correctly.